In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import time
from sklearn.model_selection import KFold

In [3]:
# Check CUDA installation and version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Nov__7_19:25:04_Pacific_Standard_Time_2025
Cuda compilation tools, release 13.1, V13.1.80
Build cuda_13.1.r13.1/compiler.36836380_0


In [4]:
# Check for GPU access
!nvidia-smi

Mon Jul  6 15:04:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.28                 Driver Version: 576.28         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   39C    P8             11W /  105W |     280MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data, data.target

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
print(f"X_train.shape:{X_train.shape}\nX_test.shape:{X_train.shape}")

X_train.shape:(16512, 8)
X_test.shape:(16512, 8)


In [8]:
# Objective function for Optuna
def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "random_state": 42,

        # Boosting
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 3000
        ),

        # Tree Complexity
        "max_depth": trial.suggest_int(
            "max_depth", 3, 8
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 15
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 3
        ),

        # Randomness
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        # Regularization
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 20, log=True
        ),
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        device="cuda",
        verbosity=0,
    )

    kf = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = []

    for train_idx, valid_idx in kf.split(X):

        X_train = X[train_idx]
        X_valid = X[valid_idx]

        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_valid)

        rmse = root_mean_squared_error(y_valid, y_pred)

        scores.append(rmse)

    return -np.mean(scores)

In [9]:
sampler = optuna.samplers.TPESampler(
    seed=42,
    multivariate=True,
    n_startup_trials=20,
    n_ei_candidates=48,
)

study = optuna.create_study(
    study_name="objective",
    direction="maximize",
    sampler=sampler,
)

C:\.venv\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


In [ ]:
start_time = time.time()
study.optimize(objective, n_trials=500, show_progress_bar=True, n_jobs= 1)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed Time: {elapsed_time:.4f} seconds")

  0%|          | 0/500 [00:00<?, ?it/s]